# List-aware ensemble release candidate

This is a copy of the existing **off-market `ensemble_release.ipynb`** workflow, adapted for **list-aware** pricing.

The same stages remain: training-only BoW feature bags and row sampling, 700-tree boosting candidates, Optuna tuning, later-month marginal ensemble admission, restartable checkpoints, frozen shadow audit, and full-history model refit.

Changes from off-market: **`log(sold_price/list_price)`** instead of direct log sale price, sale-price reconstruction using known list price, and candidate estimators from **CatBoost, XGBoost, or LightGBM** with curated feature recipes.

Use **Colab/day**, **overnight/local** or **quiet/local** profiles as in the original notebook. Switching CPU thread profiles does not change experimental model definitions. One worker at a time per checkpoint. Training and the one-way audit are independently opt-in and **disabled by default**.

Previously studied August–October 2025 months are a chronological shadow audit, not an untouched holdout. The 3.8996% earlier benchmark used a different protocol, so compare only matched-cohort results.


## 1 · One-time Colab setup

As with the off-market notebook, run the setup cells on Colab only. Use local VM storage for the live Optuna SQLite database, and a Google Drive ZIP for periodic checkpoint backup. The friend running this should use the **same dataset** and compatible Python dependencies.


In [ ]:
# Optional Google Drive mount on Colab only.
MOUNT_GOOGLE_DRIVE = False
if MOUNT_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")


In [ ]:
import os
import sys
import subprocess
from pathlib import Path

if "google.colab" in sys.modules:
    repo = Path("/content/SSU-Data-Dash-hdnl-comp")
    if not repo.exists():
        subprocess.run([
            "git", "clone",
            "https://github.com/HiddenL123/SSU-Data-Dash-hdnl-comp.git",
            str(repo)
        ], check=True)
    os.chdir(repo)
# Locate the checkout from Jupyter/VS Code, including nested notebook folders.
from pathlib import Path
import os
import sys
_start = Path(os.environ.get("DATA_DASH_ROOT", Path.cwd())).expanduser().resolve()
ROOT = next((p for p in (_start, *_start.parents)
             if (p / "data.py").is_file() and (p / "models").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Set DATA_DASH_ROOT to your Data Dash checkout before running this notebook.")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


In [ ]:
# On Colab only, install the existing repository requirements and Optuna.
# Local VS Code kernels use the repository's own environment.
if "google.colab" in sys.modules:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-r",
         str(ROOT / "requirements.txt"), "optuna", "catboost", "xgboost", "lightgbm"],
        check=True,
    )


In [ ]:
import data

homes = data.load("listings")

print("Training homes:", homes.shape)
print("Columns:", len(homes.columns))

In [ ]:
# Optional persistent checkpoint directory in Colab Drive.
if "google.colab" in sys.modules and MOUNT_GOOGLE_DRIVE:
    from pathlib import Path
    Path("/content/drive/MyDrive/automl").mkdir(parents=True, exist_ok=True)


## 2 · Choose where this session runs

Keep the **same fold protocol** when moving between Colab and local PC. The resource profile determines CPU threads only.

The list-aware factory uses Apr–May 2025 for tuning, Jun–Jul for ensemble admission, and Aug–Oct for the independently opened audit. The notebook invokes `models.listaware_ensemble_automl_factory` directly rather than the off-market CLI.


In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
from IPython.display import display

from models.automl_profiles import get_profile
from models.listaware_ensemble_automl_factory import (
    FactorySettings, FactoryRun,
)
from models.listaware_candidate_pool import CURATED_BY_FAMILY
from models.automl_portable import export_checkpoint, import_checkpoint

RELEASE_PROFILE = "overnight_local"  # "day_cloud", "overnight_local", "day_local_quiet"
RELEASE_OUTPUT_DIR = "outputs/listaware_ensemble_release_full"
RELEASE_GENERATIONS = 2             # Increase for a longer experiment BEFORE starting
RELEASE_CANDIDATES_PER_GENERATION = 3
RELEASE_OPTUNA_TRIALS = 5
RELEASE_CHECKPOINT_ARCHIVE = None   # e.g. "/content/drive/MyDrive/automl/listaware_checkpoint.zip"

profile = get_profile(RELEASE_PROFILE)
settings = FactorySettings(
    output_dir=RELEASE_OUTPUT_DIR,
    generations=RELEASE_GENERATIONS,
    candidates_per_generation=RELEASE_CANDIDATES_PER_GENERATION,
    optuna_trials=RELEASE_OPTUNA_TRIALS,
    trees=700,
    train_cap=200_000,
    validation_cap=100_000,
)
print("Execution:", profile)
print("Frozen list-aware settings:", settings)
print("Available model families:", {k: len(v) for k,v in CURATED_BY_FAMILY.items()})


In [ ]:
# Run or RESUME using the original off-market factory workflow.
# Completed folds, Optuna trials, member decisions and checkpoint archives are reused.
RUN_FULL_RELEASE_FACTORY = False

if RUN_FULL_RELEASE_FACTORY:
    factory = FactoryRun(
        settings,
        runtime_threads=profile.xgb_threads,
        checkpoint_archive=RELEASE_CHECKPOINT_ARCHIVE,
    )
    factory_result = factory.run()
    print("Finished list-aware selection:", factory_result[0].get("selection_metrics"))
    print("Selected weights:", factory_result[0].get("effective_member_weights"))


In [ ]:
# Inspect saved progress without opening audit or fitting anything.
factory_dir = Path(RELEASE_OUTPUT_DIR)
for name in ("progress.json", "roster.json", "summary.json", "events.json"):
    path = factory_dir / name
    if not path.is_file():
        continue
    payload = json.loads(path.read_text(encoding="utf-8"))
    if name == "roster.json":
        payload = {
            "processed": len(payload.get("processed", [])),
            "admitted": [
                m["candidate"]["name"] for m in payload.get("members", [])
            ],
        }
    if name == "summary.json":
        payload = {
            "selection_metrics": payload.get("selection_metrics"),
            "weights": payload.get("effective_member_weights"),
        }
    print(name, payload)


## 3 · Portable checkpoint handoff

Exactly like off-market: stop the running worker before transfer, export the output directory as a ZIP and checksum, and restore it into a new empty output folder on the other machine. Never run two processes against the same SQLite/roster state, and never keep the live SQLite DB on a mounted Drive.


In [ ]:
# Explicit checkpoint transfer; both flags default to False.
EXPORT_CHECKPOINT = False
RESTORE_CHECKPOINT = False
if EXPORT_CHECKPOINT or RESTORE_CHECKPOINT:
    if not RELEASE_CHECKPOINT_ARCHIVE:
        raise ValueError("Set RELEASE_CHECKPOINT_ARCHIVE to your ZIP path")
    if EXPORT_CHECKPOINT and RESTORE_CHECKPOINT:
        raise ValueError("Export and restore cannot happen in the same run")
    if EXPORT_CHECKPOINT:
        print("Saved checkpoint:", export_checkpoint(
            RELEASE_OUTPUT_DIR, RELEASE_CHECKPOINT_ARCHIVE,
        ))
    else:
        print("Restored checkpoint:", import_checkpoint(
            RELEASE_CHECKPOINT_ARCHIVE, RELEASE_OUTPUT_DIR,
        ))


## 4 · Freeze and evaluate (one-time)

Complete the selected number of generations first. **Opening the audit locks model membership and weights** for this output folder.

The full-history list-aware refit is a separate step after audit. It saves a model bundle under the list-aware track; **`run.py` is not modified** and does not automatically load this bundle.


In [ ]:
RUN_FULL_RELEASE_AUDIT = False  # One-way model-roster lock.
if RUN_FULL_RELEASE_AUDIT:
    frozen_factory = FactoryRun(
        settings,
        runtime_threads=profile.xgb_threads,
        checkpoint_archive=RELEASE_CHECKPOINT_ARCHIVE,
    )
    frozen_summary, audit_predictions = frozen_factory.audit()
    display(pd.DataFrame([
        {"cohort": "selection", **frozen_summary.get("selection_metrics", {})},
        {"cohort": "shadow audit", **frozen_summary.get("shadow_audit_metrics", {})},
    ]))


In [ ]:
# Full-history refit of the frozen ensemble (after locked audit).
FINALIZE_RELEASE_MODEL = False
RELEASE_MODEL_NAME = "listaware_ensemble_release"

if FINALIZE_RELEASE_MODEL:
    finalized = FactoryRun(
        settings,
        runtime_threads=profile.xgb_threads,
        checkpoint_archive=RELEASE_CHECKPOINT_ARCHIVE,
    )
    saved_location = finalized.finalize(model_name=RELEASE_MODEL_NAME)
    print("Saved list-aware model:", saved_location)
